In [ ]:
import pandas as pd

# File paths
abmil_path = r"D:\DATA\abmil_exp3.csv"
all_path = r"D:\DATA\with_snomed_category.csv"

# Load datasets
df_abmil = pd.read_csv(abmil_path)
df_all = pd.read_csv(all_path)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

for col in list_str_cols: 
    df_abmil[col] = df_abmil[col].apply(strings2lists)

In [ ]:
from helper_functions import with_tissue_artifact

# Path to cache file
cache_file = "cache_tissue_artifact.pkl"

tissue_error = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="error", version="default")

with_tissue = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="complete", version="default")
with_tissue = set(with_tissue["filename"])

error_files = set(tissue_error["filename"])
df_all_clean = df_all[~df_all["filename"].isin(error_files)].copy()

print("Original:", len(df_all))
print("After removing tissue errors:", len(df_all_clean))

In [ ]:
from helper_functions import lists2tuples

df_abmil = lists2tuples(df_abmil)

In [ ]:
# Check present T_category values in abmil_exp3
present_categories = df_abmil[df_abmil["T_category"].apply(lambda x: len(x) == 1)]["T_category"].unique()

print("T_categories in abmil_exp3:")
for cat in present_categories:
    print(cat)

In [ ]:
# Identify unique slide IDs already in abmil_exp3
existing_slides = set(df_abmil["filename"].unique())
print(len(existing_slides))

In [ ]:
from helper_functions import subset_df, subset_df_list

# Exclude slides used for training in abmil_exp3
filtered_df = df_all_clean[(~df_all_clean["filename"].isin(existing_slides))]
print("Without slides used for training: ", len(filtered_df))

# Apply HE + Hist. store filters on filtered dataset
df_HE = subset_df(filtered_df, "stain", "HE")
df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")

sampled_dfs = []

for cat in present_categories: 
    if isinstance(cat, tuple):
        cat = cat[0]
    cat = str(cat).strip()

    # All slides matching this category
    df_cat = subset_df_list(df_HE, "T_category", cat)

    # Prioritize slides by: single w/ tissue, single w/o tissue, multi w/ tissue, multi w/o tissue
    single_cat = df_cat[df_cat["T_category"].apply(lambda x: len(x) == 1)]
    single_cat_tissue = single_cat[single_cat["filename"].isin(with_tissue)]
    single_cat_no_tissue = single_cat.drop(single_cat_tissue.index)

    # Remaining multi-category slides
    multi_cat = df_cat.drop(single_cat.index)
    multi_cat_tissue = multi_cat[multi_cat["filename"].isin(with_tissue)]
    multi_cat_no_tissue = multi_cat.drop(multi_cat_tissue.index)

    # Sample up to 10 slides following the priority order
    remaining = 10
    sampled_cat = pd.DataFrame()
    groups = [single_cat_tissue, single_cat_no_tissue, multi_cat_tissue, multi_cat_no_tissue]
    for grp in groups:
        if remaining <= 0:
            break
        if len(grp) > 0:
            take = min(remaining, len(grp))
            sampled_part = grp.sample(n=take, random_state=42)
            sampled_cat = pd.concat([sampled_cat, sampled_part], ignore_index=True)
            remaining -= take
    sampled_dfs.append(sampled_cat)

df_selected = pd.concat(sampled_dfs, ignore_index=True)
df_selected = df_selected.drop_duplicates(subset="filename")

print("Final selected slides:", len(df_selected))
print("\nSlides per category:")
print(df_selected["T_category"].value_counts())

In [ ]:
# Save to csv
output_file = r"D:\DATA\abmil_inference_exp3.csv"
df_selected.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")